In [3]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import fsolve

# === Input parameters ===
E = 0.00337          # Equity value (trillions)
sigma_e = .72      # Equity volatility (85%)
r_f = 0.0429         # Risk-free rate (4.29%)
F = 3.305            # Face value of debt (trillions)
T = 10               # Time to maturity (years)

# === Step 1 & 2: Solve simultaneously for asset value V and volatility sigma_v ===
def merton_equations(vars):
    V, sigma_v = vars
    # d1 and d2 using face value F as strike
    d1 = (np.log(V / F) + (r_f + 0.5 * sigma_v**2) * T) / (sigma_v * np.sqrt(T))
    d2 = d1 - sigma_v * np.sqrt(T)
    # Equity value pulls from call option formula
    eq1 = V * norm.cdf(d1) - F * np.exp(-r_f * T) * norm.cdf(d2) - E
    # Equity vol relation
    eq2 = sigma_v * V * norm.cdf(d1) - sigma_e * E
    return [eq1, eq2]

# Initial guesses: V ≈ F+E, sigma_v small
initial_guess = [F + E, 0.01]
V, sigma_v = fsolve(merton_equations, initial_guess)

# === Step 3: Compute d1 and d2 with solved V, sigma_v ===
d1 = (np.log(V / F) + (r_f + 0.5 * sigma_v**2) * T) / (sigma_v * np.sqrt(T))
d2 = d1 - sigma_v * np.sqrt(T)

# === Step 4: Compute risky debt value D ===
F_eff = F * np.exp(-r_f * T)
D = V * norm.cdf(-d1) + F_eff * norm.cdf(d2)

# === Step 5: Implied yield and credit spread ===
y = -np.log(D / F) / T
credit_spread_bps = (y - r_f) * 10000

# === Output ===
print(f"Implied Asset Value (V):           {V:.6f} trillion")
print(f"Implied Asset Volatility (σ_V):     {sigma_v:.6f} ({sigma_v*100:.2f}%)")
print(f"Risky Debt Value (D):             {D:.6f} trillion")
print(f"Implied Debt Yield (y):            {y*100:.3f}%")
print(f"Credit Spread:                     {credit_spread_bps:.2f} bps")



Implied Asset Value (V):           1.997095 trillion
Implied Asset Volatility (σ_V):     0.016151 (1.62%)
Risky Debt Value (D):             1.993725 trillion
Implied Debt Yield (y):            5.054%
Credit Spread:                     76.43 bps
